In [5]:
import urllib.request
import os
import cv2
import matplotlib.pyplot as plt

# Ensure the model is loaded (define your model loading here)
# Example: from your_model_library import load_model
# model = load_model("path/to/your/model")

def detect_action(model, image_path):
    results = model.predict(source=image_path, conf=0.25, save=False)
    result = results[0]
    detections = [
        (model.names[int(box.cls[0])], float(box.conf[0]))
        for box in result.boxes
    ]

    def classify_action(detections):
        detected_objects = [d[0] for d in detections]
        action_scores = {
            'Stealing': 0.0,
            'Sneaking': 0.0,
            'Peaking': 0.0,
            'Normal': 0.0
        }

        if 'person' in detected_objects:
            if any(obj in detected_objects for obj in ['backpack', 'handbag', 'suitcase']):
                action_scores['Stealing'] += 0.4
            if 'refrigerator' in detected_objects:
                action_scores['Stealing'] += 0.3
            if [conf for obj, conf in detections if obj == 'person'][0] < 0.6:
                action_scores['Sneaking'] += 0.5
            if len(detected_objects) <= 2:
                action_scores['Peaking'] += 0.5

        if not any(score > 0.3 for score in action_scores.values()):
            action_scores['Normal'] = 0.4

        return action_scores

    action_scores = classify_action(detections)

    plt.figure(figsize=(15, 7))
    plt.subplot(1, 2, 1)
    img = cv2.imread(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    plt.imshow(result.plot())
    plt.title('Object Detections')
    plt.axis('off')

    plt.subplot(1, 2, 2)
    actions = list(action_scores.keys())
    scores = list(action_scores.values())
    colors = ['red' if score == max(scores) else 'blue' for score in scores]
    plt.barh(actions, scores, color=colors)
    plt.title('Action Probability Scores')
    plt.xlabel('Confidence Score')
    plt.xlim(0, 1)
    plt.tight_layout()
    plt.show()

    print("\nDetected Objects:")
    for obj, conf in detections:
        print(f"- {obj}: {conf:.2%} confidence")

    print("\nAction Analysis:")
    predicted_action = max(action_scores.items(), key=lambda x: x[1])
    print(f"Predicted Action: {predicted_action[0]} ({predicted_action[1]:.2%} confidence)")

    print("\nAll Action Scores:")
    for action, score in action_scores.items():
        print(f"- {action}: {score:.2%}")

test_urls = {
    'example_action': 'http://localhost:8889/edit/example3.jpg',
    'example_action1':'http://localhost:8889/edit/example%201.jpg',# Replace with valid URLs
}

for name, url in test_urls.items():
    try:
        print(f"\nTesting {name}:")
        image_path = f'test_{name}.jpg'

        opener = urllib.request.build_opener()
        opener.addheaders = [('User-Agent', 'Mozilla/5.0')]
        urllib.request.install_opener(opener)

        urllib.request.urlretrieve(url, image_path)
        print("Image downloaded successfully")

        # Replace with your actual model object
        # detect_action(model, image_path)

        os.remove(image_path)
    except Exception as e:
        print(f"Error processing {url}: {str(e)}")



Testing example_action:
Image downloaded successfully

Testing example_action1:
Image downloaded successfully
